# bp — MLR vs ANN Paired Statistical Comparison

This notebook compares the two "winning" models reported for bp (MLR: CW-mix
H-Included / Per(A)+A(2,2); ANN: CW-I H-Suppressed / official architecture) on
the same 66 molecules, molecule by molecule, and tests whether the difference
between them is random using paired statistical tests (Wilcoxon signed-rank
test and paired t-test). The result is written to a Word report
(`MLR_ANN_Comparison_bp.docx`).

**How it works:** for both models, a TRUE leave-one-out (LOO) refit is
performed (train on 65 molecules, predict 1, repeat 66 times). For the ANN,
this is additionally repeated end-to-end with the report's own methodology
(`SEED_LIST`, 10 seeds), and the hyperparameter search on each outer fold
(`HIDDEN_LAYER_OPTIONS` x `ALPHA_OPTIONS`, `INNER_KFOLD`-fold inner CV) is also
redone in exactly the same way as the official methodology — this is the same
logic as `nested_cross_validation_single_seed` in your ANN notebook, just run
for a SINGLE group (CW-I H-Suppressed).

**WARNING (duration):** this can take a long time (roughly 1/6 of the
"hours" the official ANN notebook reports for its 6 groups, so most likely
10-40 minutes, depending on the machine).

**Required libraries:** pandas, numpy, scikit-learn, scipy, python-docx (all
of these should already be installed if you have the ANN/MLR notebooks set
up).


**ABOUT THIS VERSION:** This is a VARIANT of the main notebook - the
literature ([36]/[37]) comparison in section 5 has been REMOVED, leaving only
the MLR vs ANN comparison (both using true LOO). Rationale: most of
[36]/[37]'s Δbp values come from their own training sets (in-sample), while
ours always come from LOO (out-of-sample), so a direct comparison was not
fair. The ORIGINAL version with the 36/37 literature comparison is kept as an
internal working copy (not included in this repository) and can be revisited
if needed again in the future.


## 1. Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import LeaveOneOut, KFold, GridSearchCV
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from docx import Document
from docx.shared import Pt, RGBColor, Inches
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_ALIGN_VERTICAL
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

print("Libraries loaded.")

## 2. Settings

This is the only place you need to change (especially `EXCEL_PATH`).

In [ ]:
EXCEL_PATH = "Alcohol_Data.xlsx"          # Path to your Excel file
OUTPUT_WORD = "MLR_ANN_Comparison_bp_MLRandANNOnly.docx"

BP_COLUMN = "bp - Boiling Point (°C)"
MOLECULE_COLUMN_PROPERTY = "Aliphatic Alcohol"    # molecule column in the All_Properties sheet
MOLECULE_COLUMN_GROUP = "Solvent"              # molecule column in the group sheets

# MLR's REPORTED winning group/descriptors for bp (do not change, fixed so
# that it exactly matches the MLR result in PAPER.docx)
MLR_GROUP_SHEET = "CW-mix_with_H"
MLR_DESCRIPTORS = ["Per(A)", "A(2,2)"]

# ANN's REPORTED best (official) group/architecture for bp
ANN_GROUP_SHEET = "CW-I_without_H"
ANN_DESCRIPTORS = ["Per(A)", "A(2,1)", "A(2,2)", "tr(A)", "epsilon(A)"]

# The SAME constants as in your ANN notebook (do not change) - the
# hyperparameter search on each outer fold is REDONE on this grid
HIDDEN_LAYER_OPTIONS = [(2,), (4,), (8,), (16,), (2, 2), (4, 4), (8, 8), (16, 16)]
ALPHA_OPTIONS = [0.0001, 0.001, 0.01, 0.1]
ANN_ACTIVATION = "relu"
ANN_SOLVER = "lbfgs"
ANN_MAX_ITER = 2000
INNER_KFOLD = 5
SEED_LIST = [0, 1, 7, 42, 100, 123, 1234, 2020, 2021, 12345]

print("Settings loaded.")

## 3. Load Data

In [ ]:
def load_data():
    all_properties = pd.read_excel(EXCEL_PATH, sheet_name="All_Properties")
    all_properties = all_properties.rename(columns={MOLECULE_COLUMN_PROPERTY: "Solvent"})
    y_all = all_properties.set_index("Solvent")[BP_COLUMN]

    mlr_df = pd.read_excel(EXCEL_PATH, sheet_name=MLR_GROUP_SHEET).set_index(MOLECULE_COLUMN_GROUP)
    ann_df = pd.read_excel(EXCEL_PATH, sheet_name=ANN_GROUP_SHEET).set_index(MOLECULE_COLUMN_GROUP)

    molecules = list(mlr_df.index)
    assert set(molecules) == set(ann_df.index) == set(y_all.index), (
        "Molecule names do not match between the MLR/ANN/All_Properties sheets!"
    )
    y = y_all.loc[molecules].values
    X_mlr = mlr_df.loc[molecules, MLR_DESCRIPTORS].values
    X_ann = ann_df.loc[molecules, ANN_DESCRIPTORS].values
    return molecules, y, X_mlr, X_ann


molecules, y, X_mlr, X_ann = load_data()
print(f"{len(molecules)} molecules loaded.")

## 4. MLR — True LOO (refit on every fold)

In [ ]:
def mlr_loo_prediction(X_mlr, y):
    loo = LeaveOneOut()
    prediction = np.zeros(len(y))
    for train_idx, test_idx in loo.split(X_mlr):
        model = LinearRegression().fit(X_mlr[train_idx], y[train_idx])
        prediction[test_idx] = model.predict(X_mlr[test_idx])
    return prediction


print("Computing MLR LOO...")
mlr_prediction = mlr_loo_prediction(X_mlr, y)

## 5. ANN — True LOO, 10-Seed Repetition

The hyperparameter search (with inner-CV) is redone on every outer fold -
exactly matching the official ANN notebook's methodology.

In [ ]:
def build_ann_pipeline(seed):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("mlp", MLPRegressor(
            activation=ANN_ACTIVATION, solver=ANN_SOLVER,
            max_iter=ANN_MAX_ITER, random_state=seed,
        )),
    ])


def ann_loo_prediction_single_seed(X_ann, y, main_seed):
    """Re-runs the hyperparameter search with inner-CV on every outer fold -
    the same logic as the official ANN notebook's
    nested_cross_validation_single_seed function."""
    loo = LeaveOneOut()
    prediction = np.zeros(len(y))
    for fold_no, (train_idx, test_idx) in enumerate(loo.split(X_ann)):
        fold_seed = main_seed * 10_000 + fold_no
        grid = {"mlp__hidden_layer_sizes": HIDDEN_LAYER_OPTIONS,
                "mlp__alpha": ALPHA_OPTIONS}
        inner_cv = KFold(n_splits=INNER_KFOLD, shuffle=True, random_state=fold_seed)
        search = GridSearchCV(build_ann_pipeline(fold_seed), grid, cv=inner_cv,
                               scoring="r2", n_jobs=-1)
        search.fit(X_ann[train_idx], y[train_idx])
        prediction[test_idx] = search.predict(X_ann[test_idx])
    return prediction


def ann_loo_prediction_multi_seed(X_ann, y):
    """NOTE: to stay consistent with the report's own method, this returns
    EACH SEED'S predictions SEPARATELY (a 10 x n matrix) - we do not average
    them FIRST to produce a single prediction/Q2. Since the official model
    is trained and used with a single seed (the 10 seeds are only for
    testing "is this result due to chance"), calculating each seed's own Q2
    separately and reporting mean+/-std correctly reflects the true usage
    scenario (and the +/-std values in the report's own tables)."""
    all_predictions = np.zeros((len(SEED_LIST), len(y)))
    for si, seed in enumerate(SEED_LIST):
        print(f"  Computing ANN LOO - seed {seed} ({si + 1}/{len(SEED_LIST)})...")
        all_predictions[si] = ann_loo_prediction_single_seed(X_ann, y, seed)
    return all_predictions


print(f"Computing ANN LOO ({len(SEED_LIST)} seeds, with hyperparameter search on every fold)...")
print("This step can take a LONG time (see the note at the top).")
ann_predictions_all_seeds = ann_loo_prediction_multi_seed(X_ann, y)  # shape: (10, 66)


## 6. Q² and Paired Statistical Tests

In [ ]:
def calculate_q2(y, prediction):
    ss_res = np.sum((y - prediction) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot


mlr_q2 = calculate_q2(y, mlr_prediction)

# ANN: a SEPARATE Q2 for EACH seed, then their mean+/-std (the report's method)
ann_q2_per_seed = np.array([calculate_q2(y, ann_predictions_all_seeds[i]) for i in range(len(SEED_LIST))])
ann_q2_mean = ann_q2_per_seed.mean()
ann_q2_std = ann_q2_per_seed.std()

print(f"MLR Q2LOO = {mlr_q2:.4f}  (value in report: 0.929)")
print(f"ANN Q2LOO (mean+/-std of the 10 seeds' individual Q2 values) = {ann_q2_mean:.4f} +/- {ann_q2_std:.4f}  (value in report: 0.964+/-0.005)")
print("Q2 per seed:", [f"{v:.4f}" for v in ann_q2_per_seed])

# For the paired test: MLR has a single set of errors (deterministic
# model). For ANN, the AVERAGE OF THE 10 SEEDS' ERRORS is used for each
# molecule (not the average of the predictions) - this measures the
# "TYPICAL error per molecule", applying the same logic at the molecule
# level.
mlr_error = y - mlr_prediction
mlr_sq_error = mlr_error ** 2
mlr_abs_error = np.abs(mlr_error)

ann_error_all_seeds = y[None, :] - ann_predictions_all_seeds  # (10, 66)
ann_sq_error_mean_per_molecule = (ann_error_all_seeds ** 2).mean(axis=0)   # (66,) - mean SQUARED error over seeds
ann_abs_error_mean_per_molecule = np.abs(ann_error_all_seeds).mean(axis=0)  # (66,) - mean ABSOLUTE error over seeds
ann_abs_error_std_per_molecule = np.abs(ann_error_all_seeds).std(axis=0)        # (66,) - std of ABSOLUTE error over seeds

w_stat, w_p = stats.wilcoxon(mlr_sq_error, ann_sq_error_mean_per_molecule)
t_stat, t_p = stats.ttest_rel(mlr_sq_error, ann_sq_error_mean_per_molecule)

# --- "ROBUST WIN" CRITERION -----------------------------------------
# Looking only at the mean can be misleading: even if ANN's mean error is
# lower than MLR's, if the variability (std) across the 10 seeds is large,
# this "win" may just be noise. So for each molecule, a WORST CASE
# (mean + std) and a BEST CASE (mean - std, not going below 0) absolute
# error range is defined for ANN:
#   - ANN is considered a "robust win" only if it is lower than MLR even
#     in its WORST CASE.
#   - MLR is considered a "robust win" only if it is lower than ANN's BEST
#     CASE.
#   - If neither holds (the ranges overlap), that molecule is considered
#     "undetermined" - i.e. ANN's seed-to-seed variability prevents a
#     definitive statement about which model is better.
ann_worst_case_abs = ann_abs_error_mean_per_molecule + ann_abs_error_std_per_molecule
ann_best_case_abs = np.maximum(0.0, ann_abs_error_mean_per_molecule - ann_abs_error_std_per_molecule)

n_ann_better = int(np.sum(ann_worst_case_abs < mlr_abs_error))
n_mlr_better = int(np.sum(mlr_abs_error < ann_best_case_abs))
n_undetermined = len(molecules) - n_ann_better - n_mlr_better

print(f"\nMean absolute error: MLR={mlr_abs_error.mean():.2f}°C  ANN={ann_abs_error_mean_per_molecule.mean():.2f}°C")
print(f"Number of molecules where ANN predicts ROBUSTLY better (even its worst case is below MLR): {n_ann_better}/{len(molecules)}")
print(f"Number of molecules where MLR predicts ROBUSTLY better (below even ANN's best case): {n_mlr_better}/{len(molecules)}")
print(f"Undetermined (ranges overlap, no clear winner): {n_undetermined}/{len(molecules)}")
print(f"\nWilcoxon signed-rank test (squared errors): p = {w_p:.4g}")
print(f"Paired t-test: t = {t_stat:.3f}, p = {t_p:.4g}")

# For the molecule table: the ANN prediction and error are shown per
# molecule as MEAN +/- STD (from each of the 10 seeds' own
# prediction/error) - this also shows which molecules ANN is consistent on
# from seed to seed, and which it is variable on.
ann_prediction_mean = ann_predictions_all_seeds.mean(axis=0)
ann_prediction_std_per_molecule = ann_predictions_all_seeds.std(axis=0)
ann_error_mean_per_molecule = ann_error_all_seeds.mean(axis=0)
ann_error_std_per_molecule = ann_error_all_seeds.std(axis=0)

# --- RELATIVE ERROR -------------------------------------------
# Definition: the values were reverse-engineered to match the style of the
# A%D column in the target tables (in references [36]/[37]):
#     Relative error = error / actual value * 100
# It is SIGNED (no absolute value is taken - if Delta bp is negative, the
# Relative error is also negative) and PERCENT-scaled (x100), but the "%"
# sign is NOT written in the table - only the number is shown (the same
# presentation style as the A%D column of reference [37]). The "actual
# value" is the experimental bp value used throughout the rest of the
# study (°C).
mlr_relative_error = mlr_error / y * 100  # (66,) - signed, percent-scaled

ann_relative_error_all_seeds = ann_error_all_seeds / y[None, :] * 100  # (10, 66) - signed
ann_relative_error_mean_per_molecule = ann_relative_error_all_seeds.mean(axis=0)
ann_relative_error_std_per_molecule = ann_relative_error_all_seeds.std(axis=0)

print(f"\nMean relative error (signed, percent-scaled, '%' not written): MLR={mlr_relative_error.mean():.2f}  ANN={ann_relative_error_mean_per_molecule.mean():.2f}")

## 7. Literature Reference Values (Δbp [36] and Δbp [37])

The dictionary below contains the Δbp values extracted from the two sources
sent by the advisor (reference 37's .doc file, reference 36's .pdf file).
Each molecule was matched by cross-checking both its name (IUPAC/common
name) and its experimental boiling point (K) - i.e. matching was not based
on name similarity alone, but also on whether that molecule's experimental
bp value in the source is consistent with the bp value in our own dataset
(converted to K).

**Δbp definition (the same in both sources):** Δbp = Experimental bp − bp
calculated by that source's own model (for 37: its own MLR model; for 36:
its own single-model prediction). Since this is a TEMPERATURE DIFFERENCE,
its NUMERICAL VALUE IS THE SAME whether computed in Kelvin or in Celsius
(the difference between the two scales is only an offset, which does not
affect differences) - so the sources using Kelvin while our dataset uses
Celsius is not a problem for the Δbp columns. (The situation may be
different for relative error/% calculations, since the percentage is
computed by dividing by the absolute temperature - this notebook always
keeps computing its own relative error values consistently in
Celsius/our own dataset.)

`None` values mean that molecule is not present in the corresponding source
(the source may not include the molecule, or it may be listed under a
different isomer/carbon count).

In [ ]:
LITERATURE_DELTA_BP = {
    # molecule_name: (Δbp[36], Δbp[37])  -- extracted from reference 37's .doc
    # and reference 36's .pdf files sent by the advisor, verified by
    # name+experimental bp matching.
    # None = this molecule is not in that source / could not be found.
    'Methanol': (None, None),
    'Ethanol': (2.35, 9.42),
    '1-Propanol': (1.46, 5.55),
    '2-Propanol': (-2.76, 10.02),
    '1-Butanol': (2.39, 2.83),
    '2-Butanol': (-5.64, -0.14),
    '2-M-1-Propanol': (2.96, 6.19),
    '2-M-2-Propanol': (-8.52, 5.25),
    '1-Pentanol': (3.44, 2.15),
    '2-Pentanol': (-5.2, -1.35),
    '3-Pentanol': (-7.01, -6.48),
    '2-M-1-Butanol': (4.77, -3.02),
    '3-M-1-Butanol': (6.5, 5.69),
    '2-M-2-Butanol': (-8.8, 0.73),
    '3-M-2-Butanol': (-3.62, 0.17),
    '2,2-MM-1-Propanol': (2.0, 8.15),
    '1-Hexanol': (3.86, 2.89),
    '2-Hexanol': (-4.19, -2.04),
    '3-Hexanol': (-6.42, -6.94),
    '2-M-1-Pentanol': (7.52, -0.54),
    '3-M-1-Pentanol': (11.51, 2.18),
    '4-M-1-Pentanol': (7.71, 4.87),
    '2-M-2-Pentanol': (-8.75, 0.8),
    '3-M-2-Pentanol': (0.3, -5.07),
    '4-M-2-Pentanol': (-3.03, 0.65),
    '2-M-3-Pentanol': (-7.92, -8.53),
    '3-M-3-Pentanol': (-6.81, -2.97),
    '2-E-1-Butanol': (8.23, 0.31),
    '2,2-M-M-1-Butanol': (None, 5.96),
    '3,3-M-M-1-Butanol': (13.15, None),
    '2,3-M-M-2-Butanol': (-6.04, -0.82),
    '3,3-M-M-2-Butanol': (-4.02, 1.31),
    '1-heptanol': (3.55, 3.99),
    '3-heptanol': (-3.6, -2.19),
    '4-heptanol': (-2.81, -2.63),
    '2-M-2-hexanol': (-18.08, 0.8),
    '3-M-3-hexanol': (-3.56, 0.35),
    '3-E-3-pentanol': (-3.33, -6.53),
    '2,2-M-M-3-pentanol': (-6.46, -4.6),
    '2,3-M-M-3-pentanol': (None, -0.9),
    '2,4-M-M-3-pentanol': (-6.34, -6.54),
    '1-octanol': (3.21, 5.51),
    '2-octanol': (-3.54, 2.32),
    '2-E-1-hexanol': (12.22, 1.09),
    '1-nonanol': (1.87, 7.08),
    '2-nonanol': (None, -1.51),
    '3-nonanol': (None, -1.27),
    '4-nonanol': (None, -1.04),
    '5-nonanol': (-2.15, 1.0),
    '2,6-M-M-4-heptanol': (None, -7.74),
    '3,5-M-M-4-heptanol': (None, -4.93),
    '3,5,5-M-M-M-1-hexanol': (None, 8.59),
    '1-decanol': (None, 8.63),
    '5-M-1-Hexanol': (6.1, 3.87),
    '5-M-2-Hexanol': (None, 1.99),
    '2-Heptanol': (-4.9, -1.2),
    '2-M-3-Heptanol': (-3.49, -2.38),
    '3-M-4-Heptanol': (-5.5, -15.23),
    '3-Octanol': (-9.05, -7.37),
    '4-Octanol': (-0.96, 0.94),
    '3-M-2-Heptanol': (None, -9.67),
    '5-M-2-Heptanol': (None, -7.81),
    '6-M-1-Heptanol': (None, 4.56),
    '4-M-1-Heptanol': (None, 1.88),
    '2,2,4-MMM-3-Pentanol': (-2.35, 8.49),
    '2-M-2-Octanol': (None, -0.22),
}


## 7b. True Validation/Test Molecules (For the Fair Sub-Comparison)

This section flags the molecules that are in [36]'s and [37]'s OWN
validation/test set (i.e. genuinely held out, out-of-sample) - the fair
sub-comparison table in Section 6 uses this.

In [ ]:
LITERATURE_FAIR_VALIDATION = {
    # molecule_name: {'has36': is this a TRUE validation-set molecule for [36],
    #                 'has37': is this a TRUE test-set molecule for [37] (for MLR)}
    # Only molecules where at least one of these two flags is True are
    # listed here - i.e. a molecule that is genuinely held out (NOT in the
    # training set) in at least one source. Other molecules (ones from the
    # training set, or not present in the source at all) are NOT in this
    # dictionary - they are skipped automatically in Section 6.
    'Ethanol': {'has36': False, 'has37': True},
    '1-Propanol': {'has36': True, 'has37': False},
    '2-M-2-Propanol': {'has36': True, 'has37': False},
    '2-Pentanol': {'has36': False, 'has37': True},
    '3-Pentanol': {'has36': True, 'has37': False},
    '2-M-1-Butanol': {'has36': False, 'has37': True},
    '2,2-MM-1-Propanol': {'has36': True, 'has37': False},
    '2-Hexanol': {'has36': False, 'has37': True},
    '2-M-1-Pentanol': {'has36': True, 'has37': False},
    '2-M-2-Pentanol': {'has36': True, 'has37': False},
    '3-M-2-Pentanol': {'has36': True, 'has37': False},
    '2,3-M-M-2-Butanol': {'has36': False, 'has37': True},
    '1-heptanol': {'has36': False, 'has37': True},
    '4-heptanol': {'has36': True, 'has37': True},
    '2-M-2-hexanol': {'has36': True, 'has37': False},
    '2,4-M-M-3-pentanol': {'has36': True, 'has37': True},
    '2,6-M-M-4-heptanol': {'has36': False, 'has37': True},
    '3,5-M-M-4-heptanol': {'has36': False, 'has37': True},
    '3,5,5-M-M-M-1-hexanol': {'has36': False, 'has37': True},
    '2-Heptanol': {'has36': True, 'has37': False},
    '2-M-3-Heptanol': {'has36': True, 'has37': False},
    '4-Octanol': {'has36': True, 'has37': False},
    '5-M-2-Heptanol': {'has36': False, 'has37': True},
    '2,2,4-MMM-3-Pentanol': {'has36': True, 'has37': False},
}

print(f"Number of true validation/test molecules: {len(LITERATURE_FAIR_VALIDATION)}")


## 8. Write Word Report

In [ ]:
NAVY = RGBColor(0x1F, 0x38, 0x64)


def _cell_background(cell, hex_color):
    shd = OxmlElement("w:shd")
    shd.set(qn("w:val"), "clear")
    shd.set(qn("w:fill"), hex_color)
    cell._tc.get_or_add_tcPr().append(shd)


def write_cell(cell, text, header=False, bold=False, align=WD_ALIGN_PARAGRAPH.CENTER):
    cell.text = ""
    p = cell.paragraphs[0]
    p.alignment = align
    cell.vertical_alignment = WD_ALIGN_VERTICAL.CENTER
    run = p.add_run(str(text))
    run.font.size = Pt(10)
    run.font.bold = header or bold
    if header:
        run.font.color.rgb = RGBColor(0xFF, 0xFF, 0xFF)
        _cell_background(cell, "4472C4")


def build_table(doc, headers, rows):
    table = doc.add_table(rows=1, cols=len(headers))
    table.style = "Table Grid"
    for i, h in enumerate(headers):
        write_cell(table.rows[0].cells[i], h, header=True)
    for row in rows:
        row_cells = table.add_row().cells
        for i, value in enumerate(row):
            # A cell can be given as a (text, is_bold) tuple - in this case
            # the cell is written in bold to highlight the winning/best
            # value. If plain text/a number is given, it is written
            # normally.
            if isinstance(value, tuple):
                text, bold = value
            else:
                text, bold = value, False
            write_cell(row_cells[i], text, bold=bold)
    return table


def add_heading(doc, text, level=1):
    h = doc.add_heading(level=level)
    run = h.add_run(text)
    run.font.color.rgb = NAVY


def add_paragraph(doc, text, bold=False):
    p = doc.add_paragraph()
    run = p.add_run(text)
    run.font.size = Pt(11)
    run.font.bold = bold
    return p


def write_report():
    doc = Document()
    add_heading(doc, "bp — MLR vs ANN Paired Statistical Comparison", level=0)
    add_paragraph(doc, "Method: official ANN methodology (hyperparameter search redone on every leave-one-out fold, averaged over 10 seeds)", bold=True)

    add_heading(doc, "1. Method", level=1)
    add_paragraph(doc,
        "For the MLR model (CW-mix H-Included, Per(A)+A(2,2)) and the ANN "
        f"model (CW-I H-Suppressed, official architecture), a true "
        f"leave-one-out (LOO) refit was performed on the same {len(molecules)} "
        "molecules: each molecule was held out in turn, the model was "
        f"retrained on the remaining {len(molecules) - 1} molecules, and a "
        "prediction was produced for the held-out molecule. For the ANN, "
        f"this process was repeated with the {len(SEED_LIST)} fixed seeds "
        "used by the report, and the final prediction per molecule was "
        "taken as the mean of these repetitions.")

    add_heading(doc, "2. Q2LOO Validation", level=1)
    build_table(doc, ["Model", "Q2LOO (this notebook)", "Q2LOO (report)"],
                  [["MLR (CW-mix H-Included)", f"{mlr_q2:.4f}", "0.929"],
                   ["ANN (CW-I H-Suppressed)", f"{ann_q2_mean:.4f} ± {ann_q2_std:.4f}", "0.975 ± 0.012"]])
    add_paragraph(doc,
        "For the ANN, Q2LOO is reported not from a single prediction set "
        f"as with MLR, but as the mean±std of the Q2 values calculated "
        f"SEPARATELY for EACH of the {len(SEED_LIST)} seeds - exactly "
        "matching the method in the official ANN report.")
    doc.add_paragraph()

    add_heading(doc, "3. Paired Statistical Test", level=1)
    build_table(doc, ["Criterion", "Value"], [
        ["Mean absolute error (MLR)", f"{mlr_abs_error.mean():.2f}°C"],
        ["Mean absolute error (ANN, 10-seed average)", f"{ann_abs_error_mean_per_molecule.mean():.2f}°C"],
        ["Number of molecules ANN predicts ROBUSTLY better", f"{n_ann_better} / {len(molecules)}"],
        ["Number of molecules MLR predicts ROBUSTLY better", f"{n_mlr_better} / {len(molecules)}"],
        ["Undetermined (ranges overlap, no clear winner)", f"{n_undetermined} / {len(molecules)}"],
        ["Wilcoxon signed-rank test (squared errors), p-value", f"{w_p:.4g}"],
        ["Paired t-test, statistic / p-value", f"t={t_stat:.3f} / p={t_p:.4g}"],
    ])
    doc.add_paragraph()
    add_paragraph(doc,
        "A \"robust\" win is determined not just from the 10 seeds' MEAN "
        "error, but by looking at the mean±std range of ANN's "
        "seed-to-seed varying error per molecule: ANN is counted as 'ANN "
        "won' only if it gives a lower error than MLR even in its WORST "
        "CASE (mean+std); MLR is counted as 'MLR won' only if it is lower "
        "than ANN's BEST CASE (mean-std) EVEN THEN. For molecules where "
        "neither of these two conditions holds (because the ranges "
        "overlap), no clear winner has been determined - this is a more "
        "cautious criterion than looking only at the means.")
    add_paragraph(doc,
        f"The ANN model gave a robustly lower absolute error than MLR on "
        f"{n_ann_better} of the {len(molecules)} molecules, MLR was "
        f"robustly better on {n_mlr_better}, and on {n_undetermined} the "
        "difference falls within ANN's seed-to-seed variability. The "
        f"paired Wilcoxon signed-rank test on squared errors gave p={w_p:.4g}, "
        f"and the paired t-test gave p={t_p:.4g}.")

    add_heading(doc, "4. Molecule-Level Results", level=1)
    rows = []
    for i, m in enumerate(molecules):
        rows.append([m, f"{y[i]:.1f}", f"{mlr_prediction[i]:.1f}",
                          f"{ann_prediction_mean[i]:.1f} ± {ann_prediction_std_per_molecule[i]:.1f}",
                          f"{y[i]-mlr_prediction[i]:.1f}",
                          f"{ann_error_mean_per_molecule[i]:.1f} ± {ann_error_std_per_molecule[i]:.1f}"])
    build_table(doc, ["Molecule", "bp (actual)", "MLR prediction", "ANN prediction (10-seed mean ± std)",
                         "MLR error", "ANN error (10-seed mean ± std)"], rows)

    add_heading(doc, "5. MLR vs ANN Comparison Table (Δbp and Relative Error)", level=1)
    # Instead of fixed numbers like "36 and 52", the number of molecules
    # coming from [36]/[37]'s own TRAINING set (in-sample) is computed
    # directly from the current molecule list and the
    # LITERATURE_FAIR_VALIDATION flags - this way, even if the dataset
    # changes (e.g. a molecule is removed/added), these numbers
    # automatically stay correct and do not need to be updated manually.
    n_36_insample = sum(
        1 for m in molecules
        if LITERATURE_DELTA_BP.get(m, (None, None))[0] is not None
        and not LITERATURE_FAIR_VALIDATION.get(m, {}).get('has36', False)
    )
    n_37_insample = sum(
        1 for m in molecules
        if LITERATURE_DELTA_BP.get(m, (None, None))[1] is not None
        and not LITERATURE_FAIR_VALIDATION.get(m, {}).get('has37', False)
    )

    add_paragraph(doc,
        "NOTE: In this version, the literature reference values ([36] and "
        "[37]) are NOT INCLUDED IN THE TABLE. Rationale: the large "
        f"majority of [36]'s and [37]'s Δbp values ({n_36_insample} and "
        f"{n_37_insample} out of {len(molecules)} molecules respectively) "
        "come from those sources' OWN TRAINING SET (in-sample, molecules "
        "already included in the model); only a minority "
        f"({len(LITERATURE_FAIR_VALIDATION)} molecules) have a true "
        f"validation/test set value. Our Δbp (MLR) and Δbp (ANN) values, "
        f"however, are a true leave-one-out (LOO) prediction for ALL "
        f"{len(molecules)} molecules, i.e. always for the held-out "
        "molecule. Because of this methodological difference, a direct "
        "Δbp comparison with [36]/[37] would not be fair, so this table "
        "only presents a comparison between ourselves (MLR vs ANN) - "
        "computed with the same method (LOO), and therefore fair.")
    add_paragraph(doc,
        "Δbp = (experimental - predicted), Relative error (%) = "
        "(experimental - predicted) / experimental x 100 - it is SIGNED "
        "(no absolute value is taken) and percent-scaled. The sign shows "
        "the direction of the model's deviation: a positive value means "
        "the model predicted BELOW the experimental value "
        "(underprediction), while a negative value means it predicted "
        "ABOVE the experimental value (overprediction).")
    add_paragraph(doc,
        "Which column is written in bold is determined with EXACTLY THE "
        "SAME logic as the 'robust win' criterion in Section 3: for ANN, "
        "not a single number but the absolute error range between the 10 "
        "seeds' BEST CASE (mean-std) and WORST CASE (mean+std) is used. "
        "ANN is made bold only if it is lower than MLR even in its WORST "
        "CASE; MLR is made bold only if it is lower than ANN's BEST CASE "
        "EVEN THEN. If neither of these two conditions holds (the ranges "
        "overlap), there is no clear winner on that row and no column is "
        "made bold ('undetermined'). Only the Δbp columns may be made "
        "bold; the Relative error columns are left untouched.")
    rows2 = []
    n_table_undetermined = 0
    for i, m in enumerate(molecules):
        # For comparison: for MLR, |value| is used directly (single-valued,
        # deterministic). For ANN, instead of a single number, the "best
        # case" (mean-std) and "worst case" (mean+std) absolute error
        # range is used - EXACTLY the same logic as the "robust win"
        # criterion in Section 3 (gives the same result as the
        # n_ann_better/n_mlr_better/n_undetermined calculation, just
        # repeated per molecule here):
        #   - ANN wins only if it is lower than MLR even in its WORST CASE.
        #   - MLR wins only if it is lower than ANN's BEST CASE even then.
        #   - If neither holds (the ranges overlap), there is NO clear
        #     winner for that molecule - no column is made bold.
        if ann_worst_case_abs[i] < mlr_abs_error[i]:
            winner = "ANN"
        elif mlr_abs_error[i] < ann_best_case_abs[i]:
            winner = "MLR"
        else:
            winner = None  # undetermined - ranges overlap, no clear winner
            n_table_undetermined += 1

        mlr_str = f"{mlr_error[i]:.1f}"
        ann_str = f"{ann_error_mean_per_molecule[i]:.1f} ± {ann_error_std_per_molecule[i]:.1f}"

        rows2.append([
            m,
            (mlr_str, winner == "MLR"),
            f"{mlr_relative_error[i]:.2f}",
            (ann_str, winner == "ANN"),
            f"{ann_relative_error_mean_per_molecule[i]:.2f} ± {ann_relative_error_std_per_molecule[i]:.2f}",
        ])
    build_table(doc, ["Aliphatic alcohols",
                         "Δbp (MLR)", "Relative error (%) (MLR)",
                         "Δbp (ANN)", "Relative error (%) (ANN)"], rows2)
    doc.add_paragraph()
    add_paragraph(doc,
        f"Note: in this {len(molecules)}-molecule table, a robust winner "
        f"between MLR and ANN could not be determined for "
        f"{n_table_undetermined} of them (the ranges overlap); no column "
        "is marked bold on these rows. (For comparison: this should match "
        f"the summary numbers in Section 3: ANN {n_ann_better}, MLR "
        f"{n_mlr_better}, undetermined {n_undetermined}.)")
    add_paragraph(doc,
        f"Mean relative error (signed, percent-scaled): MLR={mlr_relative_error.mean():.2f}  "
        f"ANN={ann_relative_error_mean_per_molecule.mean():.2f}")

    add_heading(doc, "6. Fair Sub-Comparison with Literature (Validation/Test Molecules Only)", level=1)
    add_paragraph(doc,
        "As explained in Section 5, a general literature comparison is "
        "not fair because the large majority of [36]'s and [37]'s Δbp "
        "values come from their own TRAINING set (in-sample). However, "
        "the molecules in [36]'s and [37]'s OWN reported validation/test "
        "set come from a genuinely held-out (out-of-sample) prediction - "
        "the SAME kind of comparison (measured on held-out data) as our "
        f"LOO values. This section covers ONLY these "
        f"{len(LITERATURE_FAIR_VALIDATION)} molecules: molecules flagged "
        "as being in [36]'s own validation set, or in [37]'s own test set "
        "(for MLR). If a molecule's value in the relevant source comes "
        "from the training set, that cell is left as '-' in this table "
        "(to avoid including an unfair value) - so that the whole table "
        "is an apples-to-apples comparison.")
    add_paragraph(doc,
        "In this section, MLR and ANN are compared with the literature "
        "SEPARATELY (independently of each other) - not as before, 'pick "
        "a single winner among 4 candidates'. Rationale: MLR is "
        "deterministic (no ±), so there can never be any ambiguity in the "
        "MLR-literature comparison - this prevents MLR from being "
        "unfairly marked 'undetermined' because of ANN's noise. Rule: (1) "
        "MLR is made bold if its |Δbp| is smaller than the literature's "
        "(whichever of [36] and/or [37] is available for that molecule, "
        "and if both are available, against the LOWER/HARDER one) - this "
        "comparison is always definitive. (2) The same robust criterion "
        "is applied for ANN: ANN is made bold if even its WORST CASE is "
        "lower than the literature; the literature is made bold if it is "
        "lower than even ANN's BEST CASE (and MLR also did not beat the "
        "literature); if the ranges overlap, nothing definitive can be "
        "said about ANN for that molecule. (3) The literature is made "
        "bold ONLY if it has definitively beaten BOTH MLR AND ANN - if "
        "MLR already beat the literature, or if the ANN comparison "
        "remained undetermined (it could not be confirmed that ANN "
        "definitively lost), the literature is not made bold. In this "
        "case, no column may be bold on that row - this does not mean "
        "'the literature won', it means 'no definitive result could be "
        "reached due to ANN's uncertainty'.")
    rows3 = []
    n_sub_mlr_bold = 0
    n_sub_ann_bold = 0
    n_sub_lit_bold = 0
    n_sub_none = 0
    for i, m in enumerate(molecules):
        flags = LITERATURE_FAIR_VALIDATION.get(m)
        if flags is None:
            continue
        d36, d37 = LITERATURE_DELTA_BP.get(m, (None, None))
        d36_included = d36 if flags.get('has36') else None
        d37_included = d37 if flags.get('has37') else None

        lit_candidates = {}
        if d36_included is not None:
            lit_candidates["36"] = abs(d36_included)
        if d37_included is not None:
            lit_candidates["37"] = abs(d37_included)
        best_lit_name = min(lit_candidates, key=lit_candidates.get)
        best_lit_value = lit_candidates[best_lit_name]
        # Note: when both sources are available, comparing against the
        # LOWER (harder) one is automatically the same as "beating both"
        # (beating the harder one also means beating the easier one).

        mlr_abs_i = abs(mlr_error[i])
        mlr_better = mlr_abs_i < best_lit_value  # MLR vs literature: always definitive

        if ann_worst_case_abs[i] < best_lit_value:
            ann_status = "better"
        elif best_lit_value < ann_best_case_abs[i]:
            ann_status = "worse"
        else:
            ann_status = "undetermined"

        mlr_bold = mlr_better
        ann_bold = (ann_status == "better")
        # Literature is bold only if it has definitively beaten BOTH (MLR AND ANN)
        lit_bold = (not mlr_better) and (ann_status == "worse")

        if mlr_bold:
            n_sub_mlr_bold += 1
        if ann_bold:
            n_sub_ann_bold += 1
        if lit_bold:
            n_sub_lit_bold += 1
        if not mlr_bold and not ann_bold and not lit_bold:
            n_sub_none += 1

        d36_str = f"{d36_included:.2f}" if d36_included is not None else "-"
        d37_str = f"{d37_included:.2f}" if d37_included is not None else "-"
        mlr_str = f"{mlr_error[i]:.1f}"
        ann_str = f"{ann_error_mean_per_molecule[i]:.1f} ± {ann_error_std_per_molecule[i]:.1f}"

        rows3.append([
            m,
            (d36_str, lit_bold and best_lit_name == "36"),
            (d37_str, lit_bold and best_lit_name == "37"),
            (mlr_str, mlr_bold),
            f"{mlr_relative_error[i]:.2f}",
            (ann_str, ann_bold),
            f"{ann_relative_error_mean_per_molecule[i]:.2f} ± {ann_relative_error_std_per_molecule[i]:.2f}",
        ])
    build_table(doc, ["Aliphatic alcohols", "Δbp [36] (validation)", "Δbp [37] (test)",
                         "Δbp (MLR)", "Relative error (%) (MLR)",
                         "Δbp (ANN)", "Relative error (%) (ANN)"], rows3)
    doc.add_paragraph()
    add_paragraph(doc,
        f"This fair sub-comparison contains {len(rows3)} molecules (the "
        f"only group for which the literature's genuinely held-out data "
        f"is available). Of these, MLR gave a definitively lower error "
        f"than the literature on {n_sub_mlr_bold}, ANN (even in its worst "
        f"case) gave a lower error than the literature on {n_sub_ann_bold}, "
        f"the literature definitively beat both MLR and ANN on "
        f"{n_sub_lit_bold}, and on {n_sub_none} no definitive conclusion "
        "could be reached (usually due to ANN's uncertainty) (in this "
        "last group, MLR may not have beaten the literature, but because "
        "ANN's status remained undetermined, the literature cannot be "
        "said to have definitively won either).")

    doc.save(OUTPUT_WORD)
    print(f"\nReport saved: {OUTPUT_WORD}")


write_report()
